In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from typing import List, Optional, TypedDict, Annotated
import operator
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Load environmental variables from .env
load_dotenv()

In [ ]:
from models import get_llm, Provider, ModelTier

PROVIDER = Provider.OPENAI  # swap to Provider.ANTHROPIC or Provider.GOOGLE

llm_small = get_llm(PROVIDER, ModelTier.SMALL)   # classification, clarification, store
llm_large = get_llm(PROVIDER, ModelTier.LARGE)   # SQL planning & fixing

sqlite3_path = "./northwind.db"

## **Define state and create a graph**

In [ ]:
from state import GraphState

graph = StateGraph(GraphState)




#Nodes

In [ ]:
from nodes.ingest_user_message import ingest_user_message
from nodes.question_classifier import make_question_classifier_node
from nodes.check_schema_cache import check_schema_cache
from nodes.make_schema_introspector_node import make_schema_introspector_node
from nodes.schema_normalizer import schema_normalizer
from nodes.nl_to_sql_planner import make_nl_to_sql_planner_node
from nodes.make_sql_executor_node import make_sql_executor_node
from nodes.check_execution_result import check_execution_result
from nodes.make_sql_fixer_node import make_sql_fixer_node
from nodes.make_result_summarizer_node import make_result_summarizer_node
from nodes.schema_explainer import make_schema_explainer_node
from nodes.llm_direct_answer import make_llm_direct_answer_node
from nodes.unsupported_handler import unsupported_handler
from nodes.load_knowledge_store import make_load_knowledge_store_node
from nodes.save_knowledge_store import make_save_knowledge_store_node
from nodes.detect_ambiguity import make_detect_ambiguity_node
from nodes.ask_clarification import make_ask_clarification_node
from nodes.store_clarification import make_store_clarification_node
from nodes.manage_terms import make_manage_terms_node

# Small model: fast/cheap tasks (classification, clarification)
question_classifier  = make_question_classifier_node(llm_small)
detect_ambiguity     = make_detect_ambiguity_node(llm_small)
ask_clarification    = make_ask_clarification_node(llm_small)
store_clarification  = make_store_clarification_node(llm_small)
result_summarizer    = make_result_summarizer_node(llm_small)
schema_explainer     = make_schema_explainer_node(llm_small)
llm_direct_answer    = make_llm_direct_answer_node(llm_small)
manage_terms = make_manage_terms_node(llm_small)

# Large model: SQL planning & fixing
nl_to_sql_planner = make_nl_to_sql_planner_node(llm_large)
sql_fixer         = make_sql_fixer_node(llm_large)

# Non-LLM nodes
schema_introspector  = make_schema_introspector_node(sqlite3_path)
sql_executor         = make_sql_executor_node(sqlite3_path)
load_knowledge_store = make_load_knowledge_store_node()
save_schema          = make_save_knowledge_store_node()
save_terms           = make_save_knowledge_store_node()

graph.add_node("ingest_user_message", ingest_user_message)
graph.add_node("question_classifier", question_classifier)
graph.add_node("check_schema_cache", check_schema_cache)
graph.add_node("schema_explainer", schema_explainer)
graph.add_node("schema_introspector", schema_introspector)
graph.add_node("schema_normalizer", schema_normalizer)
graph.add_node("nl_to_sql_planner", nl_to_sql_planner)
graph.add_node("sql_executor", sql_executor)
graph.add_node("check_execution_result", check_execution_result)
graph.add_node("sql_fixer", sql_fixer)
graph.add_node("result_summarizer", result_summarizer)
graph.add_node("llm_direct_answer", llm_direct_answer)
graph.add_node("unsupported_handler", unsupported_handler)
graph.add_node("load_knowledge_store", load_knowledge_store)
graph.add_node("save_schema", save_schema)
graph.add_node("save_terms", save_terms)
graph.add_node("detect_ambiguity", detect_ambiguity)
graph.add_node("ask_clarification", ask_clarification)
graph.add_node("store_clarification", store_clarification)
graph.add_node("manage_terms", manage_terms)

Edges

In [ ]:
MAX_FIX_ATTEMPTS = 2


def next_after_ingest(state: GraphState) -> str:
    """Route to clarification storage when a reply is pending, else classify."""
    if state.pending_clarification_for:
        return "store_clarification"
    return "question_classifier"


def next_after_ambiguity(state: GraphState) -> str:
    if state.ambiguous_terms:
        return "ask_clarification"
    return "check_schema_cache"


def next_after_cache(state: GraphState) -> str:
    if not state.db_schema_ready:
        return "missing"
    if state.question_type == "db_read":
        return "read_ready"
    elif state.question_type == "db_schema_question":
        return "schema_ready"
    else:
        return "read_ready"


def next_after_save_schema(state: GraphState) -> str:
    """Same routing as the old next_after_normalizer."""
    if state.question_type == "db_schema_question":
        return "schema"
    return "read"


def next_after_check(state: GraphState) -> str:
    if state.result_status == "success":
        return "success"
    elif state.fix_attempts < MAX_FIX_ATTEMPTS:
        return "fix"
    else:
        return "give_up"

In [ ]:
graph.add_edge(START, "load_knowledge_store")
graph.add_edge("load_knowledge_store", "ingest_user_message")

graph.add_conditional_edges(
    "ingest_user_message",
    next_after_ingest,
    {
        "store_clarification": "store_clarification",
        "question_classifier": "question_classifier",
    },
)

# After storing the clarification answer, persist it and re-check ambiguity
graph.add_edge("store_clarification", "save_terms")
graph.add_edge("save_terms", "detect_ambiguity")

graph.add_conditional_edges(
    "question_classifier",
    lambda s: s.question_type,
    {
        "db_read": "detect_ambiguity",
        "db_schema_question": "detect_ambiguity",
        "manage_terms": "manage_terms",
        "chitchat": "llm_direct_answer",
        "unsupported": "unsupported_handler",
    },
)

graph.add_conditional_edges(
    "detect_ambiguity",
    next_after_ambiguity,
    {
        "ask_clarification": "ask_clarification",
        "check_schema_cache": "check_schema_cache",
    },
)

# Clarification question surfaces to the user then waits for next turn
graph.add_edge("ask_clarification", END)

graph.add_conditional_edges(
    "check_schema_cache",
    next_after_cache,
    {
        "missing": "schema_introspector",
        "read_ready": "nl_to_sql_planner",
        "schema_ready": "schema_explainer",
    },
)

graph.add_edge("schema_introspector", "schema_normalizer")

# Persist schema then route to the appropriate handler
graph.add_edge("schema_normalizer", "save_schema")

graph.add_conditional_edges(
    "save_schema",
    next_after_save_schema,
    {
        "schema": "schema_explainer",
        "read": "nl_to_sql_planner",
    },
)

graph.add_edge("nl_to_sql_planner", "sql_executor")
graph.add_edge("sql_executor", "check_execution_result")

graph.add_conditional_edges(
    "check_execution_result",
    next_after_check,
    {
        "success": "result_summarizer",
        "fix": "sql_fixer",
        "give_up": "result_summarizer",
    },
)


graph.add_edge("manage_terms", END)
graph.add_edge("sql_fixer", "sql_executor")
graph.add_edge("result_summarizer", END)
graph.add_edge("schema_explainer", END)
graph.add_edge("llm_direct_answer", END)
graph.add_edge("unsupported_handler", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
import pprint
import traceback
import gradio as gr
from uuid import uuid4

def chat_fn(message: str, history: list, session_id: str):
    """
    Gradio callback.

    - message: latest user input
    - history: list of (user, bot) tuples from ChatInterface
    - session_id: unique thread id per user session
    """
    try:
        # Invoke the graph with a single new HumanMessage.
        # LangGraph will accumulate messages internally thanks to the `messages` reducer.
        result = app.invoke(
            {"messages": [HumanMessage(content=message)]},
            config={"configurable": {"thread_id": session_id}},
        )
        pprint.pp(result)
        # result is a GraphState instance → convert to dict for convenience
        answer = result.get("final_answer") or "I couldn't generate an answer."


        # ---- Debug view ----
        # Pull out interesting internal fields from GraphState
        question_type = result.get("question_type")
        db_schema_ready = result.get("db_schema_ready")
        sql_query = result.get("sql_query")
        sql_explanation = result.get("sql_explanation")
        execution_error = result.get("execution_error")
        query_metadata = result.get("query_metadata")
        assumptions = result.get("assumptions")
        db_schema_summary = result.get("db_schema_summary")

        debug_blocks = [
            "### 🐛 Debug info",
            "",
            f"**question_type**: `{question_type}`",
            f"**db_schema_ready**: `{db_schema_ready}`",
            f"**execution_error**: `{execution_error}`",
            "",
        ]

        if sql_query:
            debug_blocks.append("**Generated SQL:**")
            debug_blocks.append("```sql")
            debug_blocks.append(sql_query)
            debug_blocks.append("```")

        if sql_explanation:
            debug_blocks.append("**SQL explanation:**")
            debug_blocks.append(sql_explanation)

        if assumptions:
            debug_blocks.append("**Assumptions:**")
            for a in assumptions:
                debug_blocks.append(f"- {a}")

        if query_metadata:
            debug_blocks.append("**Query metadata:**")
            debug_blocks.append(f"- row_count: {query_metadata.get('row_count')}")
            debug_blocks.append(f"- columns: {query_metadata.get('columns')}")

        # if db_schema_summary:
        #     # Don't spam the whole thing, just the first few lines
        #     lines = db_schema_summary.splitlines()
        #     preview = "\n".join(lines[:8])
        #     debug_blocks.append("**DB schema summary (preview):**")
        #     debug_blocks.append("```text")
        #     debug_blocks.append(preview)
        #     debug_blocks.append("```")

        debug_text = "\n".join(debug_blocks)
    except Exception as e:
        # Log full traceback to your terminal
        traceback.print_exc()
        return f"Internal error: {e}"

    # Combine user-facing answer + debug panel
    return answer + "\n\n---\n\n" + debug_text

with gr.Blocks() as demo:
    gr.Markdown("# 🧠 DB QA Agent (LangGraph + Gradio + SQLite)")

    # One session_id per browser session
    session_state = gr.State(str(uuid4()))

    chat = gr.ChatInterface(
        fn=chat_fn,
        additional_inputs=[session_state],
        chatbot=gr.Chatbot(
            label="DB Agent",
            height=600
        ),
        textbox=gr.Textbox(
            placeholder="Ask things like: 'Who are the top 10 customers by revenue?'",
            label="Your question",
        ),
        title="",
        description="Ask questions about the data stored in the SQLite database.",
    )

In [ ]:
demo.launch()

In [ ]:
state = graph.__getstate__()['channels']
state['db_schema']